# Multimodal AI — Blood Work Analysis

In [1]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
from langchain.tools import tool
from langchain.agents import create_agent
import base64

load_dotenv()

True

## Encode the image and send to the vision model

In [2]:
with open("blood_work.png", "rb") as f:
    image_b64 = base64.b64encode(f.read()).decode()

llm = ChatGroq(model="meta-llama/llama-4-scout-17b-16e-instruct")

message = HumanMessage(content=[
    {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}},
    {"type": "text",      "text": "This is a blood work report. Extract all test results and flag any values outside the normal range."}
])

response = llm.invoke([message])
print(response.content)

NotFoundError: Error code: 404 - {'error': {'message': 'The model `meta-llama/llama-4-scout-17b-16e-instruct` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}

## Suggest Diet Plan Agent

The agent reads the blood work image, categorises the condition, then calls the diet tool.

In [ ]:
@tool
def get_diet_recommendation(condition: str) -> dict:
    """Given a health condition, returns a diet plan. Condition must be one of: normal, high_cholesterol, high_sugar."""
    diet_plans = {
        "high_cholesterol": {
            "eat":        ["fruits", "vegetables", "whole grains", "lean protein"],
            "do_not_eat": ["red meat", "fried food", "full-fat dairy", "processed snacks"],
        },
        "high_sugar": {
            "eat":        ["vegetables", "whole grains", "legumes", "nuts"],
            "do_not_eat": ["white rice", "white sugar", "junk food", "sugary drinks"],
        },
        "normal": {
            "eat":        ["vegetables", "fruits", "whole grains", "lean protein"],
            "do_not_eat": ["excessive sugar", "processed food", "trans fats"],
        },
    }
    return diet_plans.get(condition, diet_plans["normal"])

In [ ]:
SYSTEM_PROMPT = """
You are a helpful medical and nutrition assistant.
For the input blood work image, extract the numbers and the normal range, then categorize
the condition as one of: normal, high_cholesterol, high_sugar.
Then call the appropriate tool to retrieve and present the diet plan.
"""

diet_agent = create_agent(
    llm,
    tools=[get_diet_recommendation],
    system_prompt=SYSTEM_PROMPT,
)

In [ ]:
result = diet_agent.invoke({
    "messages": [HumanMessage(content=[
        {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}},
        {"type": "text",      "text": "Analyse this blood work report and suggest a diet plan."},
    ])]
})

print(result["messages"][-1].content)